<a href="https://colab.research.google.com/github/timosachsenberg/PyOpenMSCourse/blob/main/notebooks/PyOpenMS_Task0_Prerequisites.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install pyOpenMS (Google Colab or a local Jupyter install)
#
# The course runs on a *pinned nightly* build. The wheel is committed to this
# repository under wheels/ and fetched from an immutable git tag, so the
# notebooks keep working when the nightly server pypi.openms.de is unreachable.
# pip checks the sha256 below, so the pin is on the exact bytes, not just a name.
#
# Re-pinning is scripted - see "Updating the pinned pyOpenMS nightly" in README.md.
#
# Sources are tried in this order:
#   1. pinned wheel from this repository (exact build, hash-checked)
#   2. nightly index pypi.openms.de
#   3. stable pyopenms release on PyPI
import platform
import subprocess
import sys

PYOPENMS_VERSION = "3.6.0.dev20260903"
WHEEL_PIN_REVISION = 2  # bump when the wheel set changes for an unchanged version
WHEEL_REPO = "timosachsenberg/PyOpenMSCourse"
WHEEL_REF = f"wheels-{PYOPENMS_VERSION}-r{WHEEL_PIN_REVISION}"  # git tag holding these wheels
WHEEL_SHA256 = {
    "cp311": "25c9f994da941faf2ecba14d53b2217639e5109ba1eea2183bcf33a75c436309",
    "cp312": "b1d8f4956b93c0919957da19f2d18c20fde8f68e0ffa9d176fff8f3e663e6041",
    "cp313": "5772a06e5f24fb77e3dfda54e3022ddcb6afd5d1807f9c7f77671c9942781ad4",
}


def pip(*args, silent: bool = False) -> bool:
    """Run pip in this kernel's interpreter; return True on success."""
    out = subprocess.DEVNULL if silent else None
    return subprocess.call([sys.executable, "-m", "pip", *args],
                           stdout=out, stderr=out) == 0


def pinned_wheel_url():
    """URL of the pinned wheel for this interpreter, or None if there is none."""
    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"  # e.g. cp312
    if sys.platform != "linux" or platform.machine() != "x86_64":
        return None
    if tag not in WHEEL_SHA256:
        return None
    name = f"pyopenms-{PYOPENMS_VERSION}-{tag}-{tag}-manylinux_2_34_x86_64.whl"
    return (f"https://raw.githubusercontent.com/{WHEEL_REPO}/{WHEEL_REF}/wheels/{name}"
            f"#sha256={WHEEL_SHA256[tag]}")


def install_pyopenms() -> None:
    url = pinned_wheel_url()
    if url is not None:
        if pip("install", "-q", url, silent=True):
            return
        print("Pinned wheel unavailable here -> trying the nightly index ...")
    if pip("install", "-q", "-U", "--pre", "--extra-index-url",
           "https://pypi.openms.de/simple/", "pyopenms", silent=True):
        return
    print("Nightly index unreachable -> falling back to the stable PyPI release ...")
    pip("install", "-U", "pyopenms")


print("Installing pyOpenMS, this takes a moment ...")
install_pyopenms()
# pandas is only an optional extra of pyOpenMS, and the notebooks need it
# (for oms .get_df() and the pandas sections), so ask for it explicitly.
pip("install", "-q", "pandas", silent=True)

import pyopenms as oms

# Print what we actually ended up with - paste this if you need to report a problem.
print(f"\nPython        {sys.version.split()[0]}")
print(f"pyOpenMS      {oms.__version__}")


# Notebook 0 – Prerequisites: Python & pyOpenMS Fundamentals

**Welcome to the pyOpenMS Course!**

This optional notebook covers the foundational concepts you'll need for the main tutorials. It requires some background in Python. Don't worry if you never used Python before. Just team up with someone that has - we are sure you will learn a lot.

## Who should complete this notebook?

| Your Background | Recommendation |
|-----------------|----------------|
| New to Python | Complete Sections 1-3 |
| Know Python, new to pyOpenMS | Complete Sections 4-5 |
| Experienced in all | Skip to Notebook 1 |

## Contents

1. **Python Essentials** – Variables, lists, dictionaries, functions
2. **NumPy Basics** – Arrays and vectorized operations
3. **Pandas Basics** – DataFrames for data analysis
4. **File Formats** – FASTA, mzML, idXML, featureXML
5. **pyOpenMS Object Model** – Core classes and patterns

---

# 1. Python Essentials

This section covers the Python basics you'll encounter throughout the tutorials.

## 1.1 Variables and Basic Types

Python uses dynamic typing – you don't need to declare variable types.

In [2]:
# Numbers
mass = 1234.5678       # float (decimal number)
charge = 2             # int (integer)
mz = mass / charge     # arithmetic operations

print(f"Mass: {mass}, Charge: {charge}, m/z: {mz}")

Mass: 1234.5678, Charge: 2, m/z: 617.2839


In [3]:
# Strings
peptide = "PEPTIDER"   # text in quotes
protein = 'PROTEIN'    # single or double quotes work

# String operations
print(f"Peptide: {peptide}")
print(f"Length: {len(peptide)} amino acids")
print(f"First residue: {peptide[0]}")
print(f"Last residue: {peptide[-1]}")

Peptide: PEPTIDER
Length: 8 amino acids
First residue: P
Last residue: R


In [4]:
# Booleans
is_tryptic = True
has_modification = False

# Comparisons return booleans
print(f"Is charge > 1? {charge > 1}")
print(f"Is mass < 1000? {mass < 1000}")

Is charge > 1? True
Is mass < 1000? False


## 1.2 Lists

Lists are ordered, mutable collections. You'll use them constantly.

In [5]:
# Creating lists
masses = [500.25, 600.30, 700.35, 800.40]
amino_acids = ['A', 'R', 'N', 'D', 'C']
empty_list = []

# Accessing elements (0-indexed!)
print(f"First mass: {masses[0]}")
print(f"Last mass: {masses[-1]}")
print(f"First three: {masses[:3]}")
print(f"Last two: {masses[-2:]}")

First mass: 500.25
Last mass: 800.4
First three: [500.25, 600.3, 700.35]
Last two: [700.35, 800.4]


In [6]:
# Modifying lists
masses.append(900.45)      # Add to end
masses.insert(0, 400.20)   # Insert at position
print(f"After adding: {masses}")

# List information
print(f"Length: {len(masses)}")
print(f"Contains 600.30? {600.30 in masses}")

After adding: [400.2, 500.25, 600.3, 700.35, 800.4, 900.45]
Length: 6
Contains 600.30? True


In [7]:
# Iterating over lists
print("All masses:")
for m in masses:
    print(f"  {m}")

# With index
print("\nWith indices:")
for i, m in enumerate(masses):
    print(f"  [{i}] {m}")

All masses:
  400.2
  500.25
  600.3
  700.35
  800.4
  900.45

With indices:
  [0] 400.2
  [1] 500.25
  [2] 600.3
  [3] 700.35
  [4] 800.4
  [5] 900.45


## 1.3 List Comprehensions

A concise way to create lists. You'll see these throughout the tutorials.

In [8]:
# Traditional loop approach
squared = []
for x in [1, 2, 3, 4, 5]:
    squared.append(x ** 2)
print(f"Squared (loop): {squared}")

# List comprehension - same result, one line
squared = [x ** 2 for x in [1, 2, 3, 4, 5]]
print(f"Squared (comprehension): {squared}")

Squared (loop): [1, 4, 9, 16, 25]
Squared (comprehension): [1, 4, 9, 16, 25]


In [9]:
# With filtering
masses = [400, 500, 600, 700, 800, 900, 1000]

# Only masses > 600
large_masses = [m for m in masses if m > 600]
print(f"Large masses: {large_masses}")

# Transform and filter
mz_values = [m / 2 for m in masses if m > 600]  # Assume charge 2
print(f"m/z values (z=2, M>600): {mz_values}")

Large masses: [700, 800, 900, 1000]
m/z values (z=2, M>600): [350.0, 400.0, 450.0, 500.0]


## 1.4 Dictionaries

Key-value pairs for storing related data.

In [10]:
# Amino acid masses (monoisotopic residue masses)
aa_masses = {
    'A': 71.037,   # Alanine
    'R': 156.101,  # Arginine
    'N': 114.043,  # Asparagine
    'D': 115.027,  # Aspartic acid
    'C': 103.009,  # Cysteine
}

# Accessing values
print(f"Mass of Alanine: {aa_masses['A']} Da")
print(f"Mass of Arginine: {aa_masses['R']} Da")

# Check if key exists
print(f"\nHave mass for 'K'? {'K' in aa_masses}")

Mass of Alanine: 71.037 Da
Mass of Arginine: 156.101 Da

Have mass for 'K'? False


In [11]:
# Calculate peptide mass
peptide = "ANDR"
water_mass = 18.011  # H2O lost in peptide bond formation

# Sum residue masses + terminal H and OH
peptide_mass = sum(aa_masses[aa] for aa in peptide) + water_mass
print(f"Mass of {peptide}: {peptide_mass:.3f} Da")

Mass of ANDR: 474.219 Da


## 1.5 Functions

Reusable blocks of code.

In [12]:
def calculate_mz(mass, charge):
    """
    Calculate m/z from mass and charge.

    Parameters:
    -----------
    mass : float
        Neutral mass in Daltons
    charge : int
        Charge state (positive)

    Returns:
    --------
    float : m/z value
    """
    proton_mass = 1.00728
    return (mass + charge * proton_mass) / charge

# Use the function
print(f"m/z of 1000 Da at z=1: {calculate_mz(1000, 1):.4f}")
print(f"m/z of 1000 Da at z=2: {calculate_mz(1000, 2):.4f}")
print(f"m/z of 1000 Da at z=3: {calculate_mz(1000, 3):.4f}")

m/z of 1000 Da at z=1: 1001.0073
m/z of 1000 Da at z=2: 501.0073
m/z of 1000 Da at z=3: 334.3406


In [13]:
# Functions with default parameters
def calculate_mz(mass, charge=2):  # charge defaults to 2
    proton_mass = 1.00728
    return (mass + charge * proton_mass) / charge

print(f"Default z=2: {calculate_mz(1000):.4f}")
print(f"Override z=3: {calculate_mz(1000, charge=3):.4f}")

Default z=2: 501.0073
Override z=3: 334.3406


### Exercise 1.1: Python Basics

Complete the following exercises to test your understanding.

In [14]:
# Exercise 1.1a: Create a list of peptide sequences
peptides = ["PEPTIDE", "SEQUENCE", "EXAMPLE"]

# TODO: Use a list comprehension to get the lengths of all peptides
# lengths = ???

# Uncomment to check:
# print(f"Lengths: {lengths}")  # Should be [7, 8, 7]

In [15]:
# Exercise 1.1b: Filter peptides
# TODO: Get only peptides longer than 7 amino acids
# long_peptides = ???

# Uncomment to check:
# print(f"Long peptides: {long_peptides}")  # Should be ['SEQUENCE']

<details>
<summary><b>Click for solutions</b></summary>

```python
# 1.1a
lengths = [len(p) for p in peptides]

# 1.1b
long_peptides = [p for p in peptides if len(p) > 7]
```

</details>

---

# 2. NumPy Basics

NumPy provides efficient arrays and mathematical operations. It's used extensively in scientific Python.

In [16]:
import numpy as np

# Creating arrays
masses = np.array([400.2, 500.3, 600.4, 700.5, 800.6])
print(f"Array: {masses}")
print(f"Type: {type(masses)}")
print(f"Shape: {masses.shape}")

Array: [400.2 500.3 600.4 700.5 800.6]
Type: <class 'numpy.ndarray'>
Shape: (5,)


In [17]:
# Vectorized operations (apply to all elements at once)
mz_z2 = masses / 2  # Divide all by 2
print(f"Original masses: {masses}")
print(f"m/z at z=2: {mz_z2}")

# Much faster than loops for large arrays!

Original masses: [400.2 500.3 600.4 700.5 800.6]
m/z at z=2: [200.1  250.15 300.2  350.25 400.3 ]


In [18]:
# Useful functions
intensities = np.array([1000, 5000, 2000, 8000, 3000])

print(f"Sum: {np.sum(intensities)}")
print(f"Mean: {np.mean(intensities):.1f}")
print(f"Max: {np.max(intensities)} at index {np.argmax(intensities)}")
print(f"Min: {np.min(intensities)} at index {np.argmin(intensities)}")

Sum: 19000
Mean: 3800.0
Max: 8000 at index 3
Min: 1000 at index 0


In [19]:
# Boolean indexing (filtering)
high_intensity = intensities > 3000
print(f"Boolean mask: {high_intensity}")
print(f"High intensity values: {intensities[high_intensity]}")
print(f"Corresponding masses: {masses[high_intensity]}")

Boolean mask: [False  True False  True False]
High intensity values: [5000 8000]
Corresponding masses: [500.3 700.5]


In [20]:
# Creating special arrays
zeros = np.zeros(5)
ones = np.ones(5)
range_arr = np.arange(0, 10, 2)  # start, stop, step
linspace = np.linspace(0, 100, 5)  # start, stop, num_points

print(f"Zeros: {zeros}")
print(f"Ones: {ones}")
print(f"Range: {range_arr}")
print(f"Linspace: {linspace}")

Zeros: [0. 0. 0. 0. 0.]
Ones: [1. 1. 1. 1. 1.]
Range: [0 2 4 6 8]
Linspace: [  0.  25.  50.  75. 100.]


### Exercise 2.1: NumPy Basics

Use the `masses` and `intensities` arrays from above. Solve each task with a single vectorized expression, without writing a loop.

In [ ]:
# Exercise 2.1a: Vectorized m/z calculation
# TODO: Compute the m/z of every mass in `masses` for charge state z=3
#       (add 3 proton masses of 1.00728 Da each, then divide by 3)
# mz_z3 = ???

# Uncomment to check:
# print(f"m/z at z=3: {np.round(mz_z3, 2)}")  # Should be [134.41 167.77 201.14 234.51 267.87]

In [ ]:
# Exercise 2.1b: Boolean indexing
# TODO: Select the masses whose intensity is above the mean intensity
# above_mean = ???

# Uncomment to check:
# print(f"Masses above mean intensity: {above_mean}")  # Should be [500.3 700.5]

In [ ]:
# Exercise 2.1c: Normalize to the base peak
# Intensities are often reported relative to the most intense peak
# (the "base peak"), which is set to 100.
# TODO: Scale `intensities` so that the largest value becomes 100
# relative = ???

# Uncomment to check:
# print(f"Relative intensities: {relative}")  # Should be [ 12.5  62.5  25.  100.   37.5]

<details>
<summary><b>Click for solutions</b></summary>

```python
# 2.1a
mz_z3 = (masses + 3 * 1.00728) / 3

# 2.1b
above_mean = masses[intensities > np.mean(intensities)]

# 2.1c
relative = intensities / np.max(intensities) * 100
```

</details>

---

# 3. Pandas Basics

Pandas DataFrames are like spreadsheets in Python. You'll use them to analyze results.

In [21]:
import pandas as pd

# Creating a DataFrame
data = {
    'peptide': ['PEPTIDER', 'SAMPLER', 'TESTPEPTIDE'],
    'mass': [969.48, 787.40, 1248.59],
    'charge': [2, 2, 3],
    'score': [45.2, 32.1, 58.9]
}

df = pd.DataFrame(data)
df

,peptide,mass,charge,score
0,PEPTIDER,969.48,2,45.2
1,SAMPLER,787.40,2,32.1
2,TESTPEPTIDE,1248.59,3,58.9


In [22]:
# Accessing columns
print("Peptide column:")
print(df['peptide'])

print("\nMultiple columns:")
print(df[['peptide', 'score']])

Peptide column:
0       PEPTIDER
1        SAMPLER
2    TESTPEPTIDE
Name: peptide, dtype: str

Multiple columns:
       peptide  score
0     PEPTIDER   45.2
1      SAMPLER   32.1
2  TESTPEPTIDE   58.9


In [23]:
# Adding new columns
df['mz'] = (df['mass'] + df['charge'] * 1.00728) / df['charge']
df['length'] = df['peptide'].apply(len)  # Apply function to each row
df

,peptide,mass,charge,score,mz,length
0,PEPTIDER,969.48,2,45.2,485.747280,8
1,SAMPLER,787.40,2,32.1,394.707280,7
2,TESTPEPTIDE,1248.59,3,58.9,417.203947,11


In [24]:
# Filtering rows
high_score = df[df['score'] > 40]
print("High scoring PSMs:")
high_score

High scoring PSMs:


,peptide,mass,charge,score,mz,length
0,PEPTIDER,969.48,2,45.2,485.747280,8
2,TESTPEPTIDE,1248.59,3,58.9,417.203947,11


In [25]:
# Summary statistics
print(df.describe())

              mass    charge      score          mz     length
count     3.000000  3.000000   3.000000    3.000000   3.000000
mean   1001.823333  2.333333  45.400000  432.552836   8.666667
std     232.289953  0.577350  13.401119   47.421110   2.081666
min     787.400000  2.000000  32.100000  394.707280   7.000000
25%     878.440000  2.000000  38.650000  405.955613   7.500000
50%     969.480000  2.000000  45.200000  417.203947   8.000000
75%    1109.035000  2.500000  52.050000  451.475613   9.500000
max    1248.590000  3.000000  58.900000  485.747280  11.000000


In [26]:
# Iterating over rows
print("All PSMs:")
for idx, row in df.iterrows():
    print(f"  {row['peptide']}: score={row['score']}, m/z={row['mz']:.2f}")

All PSMs:
  PEPTIDER: score=45.2, m/z=485.75
  SAMPLER: score=32.1, m/z=394.71
  TESTPEPTIDE: score=58.9, m/z=417.20


### Exercise 3.1: Pandas Basics

Use the DataFrame `df` from above. It already has the `mz` and `length` columns.

In [ ]:
# Exercise 3.1a: Filter rows
# TODO: Select only the PSMs with charge state 2
# charge2 = ???

# Uncomment to check:
# print(f"Charge 2 peptides: {list(charge2['peptide'])}")  # Should be ['PEPTIDER', 'SAMPLER']

In [ ]:
# Exercise 3.1b: Add a column
# TODO: Add a column 'mass_per_residue' with the mass divided by the peptide length
# df['mass_per_residue'] = ???

# Uncomment to check:
# print(df['mass_per_residue'].round(2).tolist())  # Should be [121.18, 112.49, 113.51]

In [ ]:
# Exercise 3.1c: Find the best PSM
# TODO: Sort the DataFrame by score, highest first, and take the peptide of the first row
#       (hint: df.sort_values('score', ascending=False) and .iloc[0])
# best_peptide = ???

# Uncomment to check:
# print(f"Best PSM: {best_peptide}")  # Should be TESTPEPTIDE

<details>
<summary><b>Click for solutions</b></summary>

```python
# 3.1a
charge2 = df[df['charge'] == 2]

# 3.1b
df['mass_per_residue'] = df['mass'] / df['length']

# 3.1c
best_peptide = df.sort_values('score', ascending=False).iloc[0]['peptide']
# or, without sorting:
best_peptide = df.loc[df['score'].idxmax(), 'peptide']
```

</details>

---

# 4. File Formats

Common file formats you'll encounter in proteomics.

## 4.1 OpenMS File Formats

| Format | Extension | Contains |
|--------|-----------|----------|
| **idXML** | `.idXML` | Peptide/protein identifications from database search |
| **featureXML** | `.featureXML` | Detected features with quantitative information |
| **consensusXML** | `.consensusXML` | Features aligned across multiple samples |

---

# 5. pyOpenMS Object Model

pyOpenMS provides Python bindings to OpenMS, a C++ library for computational mass spectrometry.

**Online Documentation:**
- [pyOpenMS Documentation](https://pyopenms.readthedocs.io/en/latest/user_guide/index.html) – Official docs with tutorials and API
- advanced: [OpenMS C++ Class Reference](https://abibuilder.cs.uni-tuebingen.de/archive/openms/Documentation/nightly/html/index.html) – Detailed API reference

In [27]:
import pyopenms as oms
print(f"pyOpenMS version: {oms.__version__}")

pyOpenMS version: 3.6.0.dev20260828


## 5.1 Getting Help and Documentation

pyOpenMS has extensive documentation. Here's how to explore it:

In [28]:
# List all available methods on a class using dir()
# Filter to show only public methods (not starting with '_')
spectrum = oms.MSSpectrum()
public_methods = [m for m in dir(spectrum) if not m.startswith('_')]
print("MSSpectrum methods (first 15):")
print(public_methods[:15])

MSSpectrum methods (first 15):
['MSSpectrumRasterAggregation', 'calculateTIC', 'clear', 'clearMetaInfo', 'clearRanges', 'comment', 'containsIMData', 'df_columns', 'drift_time', 'drift_time_array_view', 'drift_time_unit', 'findHighestInWindow', 'findNearest', 'float_data_array_view', 'float_data_array_views']


In [ ]:
# Get documentation for a specific method using help()
# This shows the docstring with parameter info and return types.
#
# Ask the CLASS for the method (oms.MSSpectrum.setRT), not an instance
# (spectrum.setRT). pyOpenMS is built with nanobind, and a method looked up on
# an instance comes back as a bound-method wrapper that help() does not
# recognise as a function - it would document the wrapper instead of setRT.
help(oms.MSSpectrum.setRT)

In [ ]:
# The docstring itself is always reachable, on an instance too:
print(spectrum.setRT.__doc__)

In [30]:
# show the full help
help(oms.AASequence)

Help on class AASequence in module pyopenms._pyopenms_chemistry:

class AASequence(builtins.object)
 |  AASequence(*args, **kwargs)
 |
 |  Representation of a peptide/protein sequence
 |  This class represents amino acid sequences in OpenMS. An AASequence
 |  instance primarily contains a sequence of residues.
 |
 |  Methods defined here:
 |
 |  __add__(...)
 |      __add__(self, arg: pyopenms._pyopenms_chemistry.AASequence, /) -> pyopenms._pyopenms_chemistry.AASequence
 |
 |  __copy__(...)
 |      __copy__(self) -> pyopenms._pyopenms_chemistry.AASequence
 |
 |  __deepcopy__(...)
 |      __deepcopy__(self, memo: dict) -> pyopenms._pyopenms_chemistry.AASequence
 |
 |  __eq__(...)
 |      __eq__(self, arg: pyopenms._pyopenms_chemistry.AASequence, /) -> bool
 |
 |  __getitem__(...)
 |      __getitem__(self, i: int) -> pyopenms._pyopenms_chemistry.Residue
 |
 |      Returns a copy of the residue at index i
 |
 |  __hash__(...)
 |      __hash__(self) -> int
 |
 |  __iadd__(...)
 |      __ia

## 5.2 Common Pattern: Load → Process → Save

In [31]:
# The standard pyOpenMS workflow pattern:

# 1. Create empty container
exp = oms.MSExperiment()

# 2. Load data from file
# oms.MzMLFile().load("data.mzML", exp)

# 3. Process data
# ... (e.g., feature detection, filtering)

# 4. Save results
# oms.MzMLFile().store("output.mzML", exp)

print("Pattern: Create → Load → Process → Save")

Pattern: Create → Load → Process → Save


## 5.3 Working with Algorithms

In [ ]:
# Most algorithms follow this pattern:

# 1. Create algorithm instance
tsg = oms.TheoreticalSpectrumGenerator()

# 2. Inspect the parameters (names and current values)
params = tsg.getParameters()
print(params.asDict())  # Show all parameters as a plain Python dictionary

# 3. Modify parameters and hand them back to the algorithm
params.setValue("add_b_ions", "true")
params.setValue("add_y_ions", "true")
tsg.setParameters(params)

# 4. Run the algorithm - named arguments make the call self-explanatory
peptide = oms.AASequence("PEPTIDER")
theo_spectrum = oms.MSSpectrum()
tsg.getSpectrum(theo_spectrum, peptide, min_charge=1, max_charge=2)

print(f"\nGenerated {theo_spectrum.size()} theoretical peaks")

{'isotope_model': 'none', 'max_isotope': 2, 'max_isotope_probability': 0.05, 'add_metainfo': 'false', 'add_losses': 'false', 'add_term_losses': 'false', 'add_internal_fragments': 'false', 'sort_by_position': 'true', 'add_precursor_peaks': 'false', 'add_all_precursor_charges': 'false', 'add_abundant_immonium_ions': 'false', 'add_first_prefix_ion': 'false', 'add_y_ions': 'true', 'add_b_ions': 'true', 'add_a_ions': 'false', 'add_c_ions': 'false', 'add_x_ions': 'false', 'add_z_ions': 'false', 'add_zp1_ions': 'false', 'add_zp2_ions': 'false', 'y_intensity': 1.0, 'b_intensity': 1.0, 'a_intensity': 1.0, 'c_intensity': 1.0, 'x_intensity': 1.0, 'z_intensity': 1.0, 'relative_loss_intensity': 0.1, 'precursor_intensity': 1.0, 'precursor_H2O_intensity': 1.0, 'precursor_NH3_intensity': 1.0}

Generated 26 theoretical peaks


Parameters live in a `Param` object. `asDict()` gives a quick overview; `keys()`, `getValue()` and `getDescription()` let you look at one parameter at a time. The description tells you what a parameter does, which is the first thing to check when you meet a new algorithm.

Also note the **named arguments** in `getSpectrum(...)` above. pyOpenMS methods accept keyword arguments (releases up to 3.5 only accept positional ones), so `min_charge=1, max_charge=2` reads much better than a bare `1, 2`. Use `help(oms.TheoreticalSpectrumGenerator.getSpectrum)` to see the argument names of any method.

In [ ]:
# Every parameter also carries a description - print the first few
for key in params.keys()[:6]:
    print(f"{key:24s} = {params.getValue(key)!s:8s} {params.getDescription(key)}")

isotope_model            = none     Model to use for isotopic peaks ('none' means no isotopic peaks are added, 'coarse' adds isotopic peaks in unit mass distance, 'fine' uses the hyperfine isotopic generator to add accurate isotopic peaks. Note that adding isotopic peaks is very slow.
max_isotope              = 2        Defines the maximal isotopic peak which is added if 'isotope_model' is 'coarse'
max_isotope_probability  = 0.05     Defines the maximal isotopic probability to cover if 'isotope_model' is 'fine'
add_metainfo             = false    Adds the type of peaks as metainfo to the peaks, like y8+, [M-H2O+2H]++
add_losses               = false    Adds common losses to those ion expect to have them, only water and ammonia loss is considered
add_term_losses          = false    Adds common N- and C-term losses (only if add_losses=true and isotope_model=none), only water and ammonia loss is considered.
